<a href="https://colab.research.google.com/github/sanjeetworld/multilingual-vlm-qa-pipeline/blob/main/main-code-content.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Cell 1 — Setup

In [ ]:
!unzip /content/batch_001.zip -d /content/images/

Archive:  /content/batch_001.zip
replace /content/images/arc_0000017.webp? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

#### Cell 2 — Install deps

In [ ]:
!pip install -q transformers accelerate qwen-vl-utils pillow requests tqdm
print('✅ Dependencies installed')

#### Cell 3 — Imports & Config

In [ ]:
import os, json, time, glob, traceback
from pathlib import Path
from datetime import datetime
from tqdm.notebook import tqdm
from PIL import Image
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
from qwen_vl_utils import process_vision_info

# —— CONFIGURATION ——
JSONL_PATH     = '/content/images_scripts_multilingual.jsonl'
IMAGE_DIR      = '/content/images'
OUTPUT_DIR     = '/content/output'
MODEL_ID       = 'Qwen/Qwen2.5-VL-3B-Instruct'
MAX_NEW_TOKENS = 500

os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(IMAGE_DIR, exist_ok=True)

if os.path.exists(IMAGE_DIR):
    imgs = [f for f in os.listdir(IMAGE_DIR)
            if f.lower().endswith(('.jpg','.jpeg','.png','.gif','.webp'))]
    print(f'✅ Found {len(imgs)} images in {IMAGE_DIR}')
print(f'✅ Config OK — output: {OUTPUT_DIR}')

#### Cell 4 — Load model

In [ ]:
print(f'Loading {MODEL_ID} ...')
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_ID, torch_dtype=torch.float16, device_map='auto'
)
model.eval()
processor = AutoProcessor.from_pretrained(MODEL_ID)
device = next(model.parameters()).device
print(f'\n✅ Model loaded on: {device}')
if torch.cuda.is_available():
    print(f'   VRAM used: {torch.cuda.memory_allocated()/1e9:.1f} GB')

#### Cell 5 — Image loader helper

In [ ]:
import glob as _glob

def load_image(img_id: str) -> Image.Image | None:
    """Load image from IMAGE_DIR using exact image ID."""
    for ext in ['jpg','jpeg','JPG','JPEG','png','PNG','gif','webp']:
        path = os.path.join(IMAGE_DIR, f'{img_id}.{ext}')
        if os.path.exists(path):
            try:
                img = Image.open(path).convert('RGB')
                img.thumbnail((1024, 1024), Image.LANCZOS)
                return img
            except Exception as e:
                print(f'   ❌ Cannot open {path}: {e}')
                return None
    return None

#### Cell 6 — Prompt builder (Task 6: Multilingual & Code-Switched QA)

In [ ]:
def build_prompt(record: dict) -> str:
    img_id  = record.get('id', 'unknown')
    caption = (record.get('caption') or '').strip()
    caption_context = ''
    if caption:
        caption_context = f'Dataset caption (use as cultural context):\n"{caption}"\n\n'

    return f"""You are an expert in Indian languages, culture, and society.

{caption_context}Analyze the image carefully and generate ONE question-answer pair for EACH of the following three types. Return ONLY the structured output below.

Image ID: {img_id}

a. CODE-SWITCHED QA (mix English and an Indian language):
Q: <question>
A: <answer>

b. TRANSLITERATION-AWARE QA (Romanized Indian language, no native script):
Q: <question>
A: <answer>

c. INDIC LANGUAGE QA (fully in native script of a regional language):
LANGUAGE: <regional language name>
Q: <question>
A: <answer>"""

#### Cell 7 — Inference function

In [ ]:
def run_inference(image: Image.Image, prompt: str) -> str:
    messages = [{
        'role': 'user',
        'content': [
            {'type': 'image', 'image': image},
            {'type': 'text',  'text': prompt},
        ],
    }]
    text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(
        text=[text], images=image_inputs, videos=video_inputs,
        padding=True, return_tensors='pt',
        min_pixels=128*28*28, max_pixels=256*28*28,
    ).to(device)
    with torch.no_grad():
        generated_ids = model.generate(**inputs, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    trimmed = [o[len(i):] for i, o in zip(inputs.input_ids, generated_ids)]
    return processor.batch_decode(trimmed, skip_special_tokens=True,
                                   clean_up_tokenization_spaces=False)[0].strip()

#### Cell 8 — Output parser

In [ ]:
def parse_output(raw: str, record: dict) -> dict:
    def _extract(text, start_marker, end_markers):
        si = text.find(start_marker)
        if si == -1: return ''
        si += len(start_marker)
        ei = len(text)
        for em in end_markers:
            idx = text.find(em, si)
            if idx != -1 and idx < ei: ei = idx
        return text[si:ei].strip()

    a_block = _extract(raw, 'a. CODE-SWITCHED QA', ['b. TRANSLITERATION-AWARE QA'])
    b_block = _extract(raw, 'b. TRANSLITERATION-AWARE QA', ['c. INDIC LANGUAGE QA'])
    c_block = _extract(raw, 'c. INDIC LANGUAGE QA', [])

    return {
        # —— Identity fields ——
        'id':               record.get('id', ''),
        'image_url':        record.get('image_url', ''),
        'category':         record.get('category', ''),
        'source':           record.get('source', ''),
        'license':          record.get('license', ''),
        'original_caption': record.get('caption', ''),
        # —— Task 6 QA fields ——
        'code_switched_q':   a_block.split('A:')[0].replace('Q:', '').strip(),
        'code_switched_a':   a_block.split('A:')[-1].strip(),
        'transliteration_q': b_block.split('A:')[0].replace('Q:', '').strip(),
        'transliteration_a': b_block.split('A:')[-1].strip(),
        'indic_language':    _extract(c_block, 'LANGUAGE:', ['Q:']).strip(),
        'indic_q':           c_block.split('Q:')[-1].split('A:')[0].strip(),
        'indic_a':           c_block.split('A:')[-1].strip(),
        # —— Meta ——
        'raw_output':   raw,
        'status':       'success',
        'annotated_at': datetime.now().isoformat(),
    }

print('✅ Helper functions defined')

#### Cell 9 — Load dataset & build lookup

In [ ]:
records = []
with open(JSONL_PATH, 'r', encoding='utf-8') as f:
    for line in f:
        line = line.strip()
        if line:
            records.append(json.loads(line))

records_by_id = {r['id']: r for r in records}
print(f'Total records in dataset : {len(records):,}')
print(f'Sample record keys       : {list(records[0].keys())}')

#### Cell 10 — Resume setup & discover images


In [ ]:
OUTPUT_JSON = os.path.join(OUTPUT_DIR, 'qa_annotations.json')

annotated = {}
if os.path.exists(OUTPUT_JSON):
    with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
        existing = json.load(f)
    annotated = {item['id']: item for item in existing if item.get('status') == 'success'}
    print(f'Already annotated: {len(annotated):,} images (will skip these)')
else:
    print('No existing annotations found — starting fresh')

image_files = _glob.glob(os.path.join(IMAGE_DIR, '*'))
image_ids_available = []
for fp in image_files:
    fname = os.path.basename(fp)
    img_id = os.path.splitext(fname)[0]
    if img_id in records_by_id:
        image_ids_available.append(img_id)

to_process = [img_id for img_id in image_ids_available if img_id not in annotated]

print(f'Images in folder       : {len(image_ids_available)}')
print(f'Already annotated      : {len(annotated):,}')
print(f'To process this session: {len(to_process)}')

#### Cell 11 — Main processing loop

In [ ]:
results = dict(annotated)
failed_ids = []
new_count = 0

for img_id in tqdm(to_process, desc='Generating multilingual QA'):
    record = records_by_id[img_id]

    if torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.synchronize()

    image = load_image(img_id)
    if image is None:
        failed_ids.append(img_id)
        results[img_id] = {
            'id': img_id, 'image_url': record.get('image_url', ''),
            'status': 'image_not_found', 'annotated_at': datetime.now().isoformat()
        }
        continue

    prompt = build_prompt(record)

    try:
        raw_output = run_inference(image, prompt)
    except Exception as e:
        print(f'\n❌ Inference error [{img_id}]: {e}')
        traceback.print_exc()
        failed_ids.append(img_id)
        results[img_id] = {
            'id': img_id, 'image_url': record.get('image_url', ''),
            'status': 'inference_error', 'error': str(e),
            'annotated_at': datetime.now().isoformat()
        }
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        continue

    parsed = parse_output(raw_output, record)
    results[img_id] = parsed
    new_count += 1

    with open(OUTPUT_JSON, 'w', encoding='utf-8') as f:
        json.dump(list(results.values()), f, ensure_ascii=False, indent=2)

print(f"\n{'='*55}")
print('Session complete!')
print(f'✅ Newly annotated : {new_count}')
print(f'📦 Total in file   : {len(results):,}')
print(f'❌ Failed          : {len(failed_ids)}')
print(f'💾 Saved to        : {OUTPUT_JSON}')

#### Cell 12 — Verify output

In [ ]:
import textwrap

with open(OUTPUT_JSON, 'r', encoding='utf-8') as f:
    all_annotations = json.load(f)

success = [a for a in all_annotations if a.get('status') == 'success']
print(f'Total annotations: {len(all_annotations):,}  |  Successful: {len(success):,}')
print()

for r in success[:3]:
    print('=' * 65)
    print(f"ID          : {r['id']}")
    print(f"Category    : {r.get('category','')}")
    print(f"\nCODE-SWITCHED Q: {r.get('code_switched_q','')}")
    print(f"CODE-SWITCHED A: {r.get('code_switched_a','')}")
    print(f"\nTRANSLIT Q     : {r.get('transliteration_q','')}")
    print(f"TRANSLIT A     : {r.get('transliteration_a','')}")
    print(f"\nINDIC LANG     : {r.get('indic_language','')}")
    print(f"INDIC Q        : {r.get('indic_q','')}")
    print(f"INDIC A        : {r.get('indic_a','')}")
    print()

#### Cell 13 — Download

In [ ]:
from google.colab import files
files.download(OUTPUT_JSON)
print('✅ Download started')